# Pretrain IndPatchTST


In [1]:
import importlib.util
from pathlib import Path
import sys

# Fallback: if package not installed in this kernel, add repo root to sys.path
if importlib.util.find_spec('src') is None:
    ROOT = Path('..').resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))



**Prerequisite:** install the package in editable mode from the repo root:

```bash
pip install -e .
```


In [2]:
from pathlib import Path
import torch
import torch.nn as nn
from src.data.dataloader import build_etth1_dataloaders
from src.models.indpatchtst import build_model_from_config
from src.training.optuna_search import bayesian_search
from src.training.train_indpatchtst_reg import train_and_valid_loop

data_path = Path('../data/ETTh1.csv')
if not data_path.exists():
    raise FileNotFoundError('Place ETTh1.csv in ../data/')

WINDOW, HORIZON = 36, 24
train_dl, val_dl, n_features = build_etth1_dataloaders(
    str(data_path), window=WINDOW, horizon=HORIZON, batch_size=128
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Recherche bayesienne (reduis n_trials pour un test rapide)
best_params, best_loss = bayesian_search(
    train_dl, val_dl, WINDOW, HORIZON, device, n_trials=10, max_epochs=5
)

# Re-entrainement long avec la meilleure config
model = build_model_from_config(best_params, n_features, WINDOW, HORIZON).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=best_params['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
criterion = nn.MSELoss()
train_and_valid_loop(
    model, train_dl, val_dl, opt, criterion, n_epochs=10, device=device, scheduler=scheduler
)


[I 2026-03-12 12:16:19,248] A new study created in memory with name: no-name-5e3729bf-f3de-4be3-a7e0-37eb01f1ece1
c:\Users\elyan\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(
[I 2026-03-12 12:16:37,566] Trial 0 finished with value: 0.23560450969415744 and parameters: {'d_model': 128, 'n_heads': 1, 'patch_len': 11, 'stride': 3, 'n_layers': 5, 'd_ff': 512, 'dropout': 0.10308477766956904, 'revin': False, 'lr': 0.00020298058052421552}. Best is trial 0 with value: 0.23560450969415744.
c:\Users\elyan\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(
c:\Users\elyan\AppData\Local\Programs\Python\Python311\Lib\site-pa


=== Meilleure config bayésienne ===
{'d_model': 256, 'n_heads': 8, 'patch_len': 11, 'stride': 4, 'n_layers': 2, 'd_ff': 256, 'dropout': 0.0799663418334207, 'revin': False, 'lr': 0.00022150109033839915}
Meilleure valid loss: 0.2001
Epoch 01 | train=0.5211 | valid=0.2784
Epoch 02 | train=0.3615 | valid=0.2423
Epoch 03 | train=0.3071 | valid=0.2741
Epoch 04 | train=0.2658 | valid=0.2016
Epoch 05 | train=0.2386 | valid=0.1895
Epoch 06 | train=0.2180 | valid=0.2272
Epoch 07 | train=0.1999 | valid=0.1490
Epoch 08 | train=0.1881 | valid=0.1618
Epoch 09 | train=0.1743 | valid=0.1828
Epoch 10 | train=0.1661 | valid=0.1788


{'train_loss': [0.5211303039097757,
  0.36153597775716523,
  0.3071125357915751,
  0.2657982958937753,
  0.23855090197993173,
  0.21797444884488604,
  0.1998988064477773,
  0.18813159162204274,
  0.17430958380537315,
  0.1660621239935071],
 'valid_loss': [0.2783957500134763,
  0.24228772671570323,
  0.2740808971713796,
  0.20162376701558088,
  0.18948752279944042,
  0.22715071315880347,
  0.14898309245038388,
  0.16178121687213534,
  0.18276112118626844,
  0.17878468538261308]}